# S3.2 — ArXiv API Client

Interactive verification of the arXiv API client: schema, query building, fetch papers, PDF download, and factory.

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. ArxivPaper Schema

In [2]:
from src.schemas.arxiv import ArxivPaper

paper = ArxivPaper(
    arxiv_id="2401.12345",
    title="Attention Is All You Need",
    authors=["Vaswani", "Shazeer"],
    abstract="We propose a new architecture based on attention.",
    categories=["cs.AI", "cs.CL"],
    published_date="2024-01-15T00:00:00Z",
    pdf_url="https://arxiv.org/pdf/2401.12345.pdf",
)

assert paper.arxiv_id == "2401.12345"
assert paper.arxiv_url == "https://arxiv.org/abs/2401.12345"
assert paper.updated_date is None
print(f"ArxivPaper: {paper.title}")
print(f"URL: {paper.arxiv_url}")
print("\n\u2713 ArxivPaper schema works correctly")

ArxivPaper: Attention Is All You Need
URL: https://arxiv.org/abs/2401.12345

✓ ArxivPaper schema works correctly


## 2. ArxivClient — Query Building

In [3]:
from src.services.arxiv.client import ArxivClient

client = ArxivClient(cache_dir="/tmp/test_arxiv_nb")

q1 = client._build_query(category="cs.AI")
assert q1 == "cat:cs.AI"
print(f"Category only: {q1}")

q2 = client._build_query(from_date="20240101", to_date="20240131")
assert "submittedDate:[20240101 TO 20240131]" in q2
print(f"Date range: {q2}")

q3 = client._build_query(category="cs.CL", search_query="transformers")
assert "cat:cs.CL" in q3 and "all:transformers" in q3
print(f"Combined: {q3}")

print("\n\u2713 Query building works correctly")

Category only: cat:cs.AI
Date range: cat:cs.AI AND submittedDate:[20240101 TO 20240131]
Combined: cat:cs.CL AND all:transformers

✓ Query building works correctly


## 3. Rate Limiting

In [4]:
assert client.rate_limit_delay >= 3.0, "Rate limit must be >= 3s"

# Test that setting delay < 3.0 gets clamped
c2 = ArxivClient(rate_limit_delay=1.0, cache_dir="/tmp/test_arxiv_nb2")
assert c2.rate_limit_delay >= 3.0

print(f"Rate limit delay: {client.rate_limit_delay}s (min 3.0s enforced)")
print("\n\u2713 Rate limiting enforced correctly")

Rate limit delay: 3.0s (min 3.0s enforced)

✓ Rate limiting enforced correctly


## 4. Factory

In [5]:
from src.services.arxiv.factory import make_arxiv_client

make_arxiv_client.cache_clear()
c1 = make_arxiv_client()
c2 = make_arxiv_client()

assert isinstance(c1, ArxivClient)
assert c1 is c2, "Factory must return cached singleton"
make_arxiv_client.cache_clear()

print(f"Factory created ArxivClient with base_url={c1.base_url}")
print(f"Category: {c1.search_category}, Max results: {c1.max_results}")
print("\n\u2713 Factory creates and caches ArxivClient correctly")

Factory created ArxivClient with base_url=https://export.arxiv.org/api/query
Category: cs.AI, Max results: 100

✓ Factory creates and caches ArxivClient correctly


## 5. Fetch Papers (mocked)

In [6]:
import asyncio
from unittest.mock import AsyncMock

SAMPLE_FEED = """<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom">
  <entry>
    <id>http://arxiv.org/abs/2401.12345v1</id>
    <title>Test Paper Title</title>
    <summary>Test abstract.</summary>
    <published>2024-01-15T00:00:00Z</published>
    <updated>2024-01-15T00:00:00Z</updated>
    <author><name>Author A</name></author>
    <category term="cs.AI"/>
  </entry>
</feed>"""

async def test_fetch():
    c = ArxivClient(cache_dir="/tmp/test_arxiv_nb3")
    c._make_request = AsyncMock(return_value=SAMPLE_FEED)
    papers = await c.fetch_papers(max_results=5)
    assert len(papers) == 1
    assert papers[0].arxiv_id == "2401.12345"
    assert papers[0].title == "Test Paper Title"
    return papers

papers = await test_fetch()
print(f"Fetched {len(papers)} paper(s): {papers[0].title}")
print(f"arXiv ID: {papers[0].arxiv_id}")
print(f"PDF URL: {papers[0].pdf_url}")
print("\n\u2713 fetch_papers works correctly (mocked)")

Fetched 1 paper(s): Test Paper Title
arXiv ID: 2401.12345
PDF URL: https://arxiv.org/pdf/2401.12345.pdf

✓ fetch_papers works correctly (mocked)


## 6. Summary

All S3.2 components verified:
- ArxivPaper schema with arxiv_url property
- Query building (category, date range, search terms)
- Rate limiting (>= 3s enforced)
- Factory (singleton caching)
- fetch_papers (Atom XML parsing)